# FruitCount — Pipeline Kaggle lấy source từ GitHub

Giữ nguyên pipeline chuẩn bị dữ liệu, huấn luyện, resume checkpoint, đánh giá và so sánh của kaggle_pipeline.ipynb.

Trước khi chạy:
1. Commit và push source lên nhánh TLuan của repository GitHub.
2. Bật Internet và GPU trong Settings của notebook Kaggle.
3. Add Input bộ MinneApple và sửa đường dẫn ảnh/mask ở cell cấu hình dữ liệu.

Source được clone vào /kaggle/working/FruitCountGit. Không cần upload source thành Kaggle Dataset.
Checkpoint được lưu vào /kaggle/working/runs sau mỗi epoch. Khi đổi phiên, cần lưu output phiên trước và khôi phục runs để resume.
Clone đã có được dùng lại, không tự pull. Muốn lấy code mới, dùng phiên mới hoặc đổi WORK sang thư mục chưa tồn tại.


In [7]:
import os, shutil, sys, subprocess
from pathlib import Path

# Use a conservative working-directory budget, distinct from GPU VRAM/RAM.
from pathlib import Path
WORKING_ROOT = Path('/kaggle/working')
if not WORKING_ROOT.is_dir():
    raise RuntimeError('This notebook targets Kaggle; /kaggle/working is unavailable.')
os.environ['FRUITCOUNT_WORKING_ROOT'] = str(WORKING_ROOT)
os.environ['FRUITCOUNT_DISK_BUDGET_GB'] = '18'
os.environ['FRUITCOUNT_MIN_FREE_GB'] = '2'
os.environ['HF_HOME'] = str(WORKING_ROOT / '.cache/huggingface')
os.environ['TORCH_HOME'] = str(WORKING_ROOT / '.cache/torch')
os.environ['HF_HUB_DISABLE_XET'] = '1'
os.environ['PIP_NO_CACHE_DIR'] = '1'
print('Working disk free (GB):', shutil.disk_usage(WORKING_ROOT).free / 10**9)

REPO_URL = 'https://github.com/QNhut/BTL_CV.git'
BRANCH = 'TLuan'
WORK = '/kaggle/working/FruitCountGit'

work_dir = Path(WORK)
if not work_dir.exists():
    subprocess.run(
        ['git', 'clone', '--depth', '1', '--branch', BRANCH, '--single-branch', REPO_URL, WORK],
        check=True,
    )
else:
    if not (work_dir / '.git').is_dir():
        raise RuntimeError(f'{WORK} exists but is not a Git checkout. Choose another WORK directory.')
    current_branch = subprocess.check_output(
        ['git', '-C', WORK, 'branch', '--show-current'], text=True
    ).strip()
    if current_branch != BRANCH:
        raise RuntimeError(f'Existing checkout uses {current_branch}, expected {BRANCH}. Choose another WORK directory.')

for required in ('src', 'scripts/train.py', 'configs/dataset.yaml', 'requirements.txt'):
    if not (work_dir / required).exists():
        raise FileNotFoundError(f'Missing project file: {required}')

os.chdir(WORK)
if WORK not in sys.path:
    sys.path.insert(0, WORK)
print('Source:', REPO_URL, '| branch:', BRANCH)
subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
subprocess.run([sys.executable, '-c', "import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())"], check=True)


Working disk free (GB): 20.929298432
Source: https://github.com/QNhut/BTL_CV.git | branch: TLuan
02996fe
torch 2.11.0+cu128 | cuda True


CompletedProcess(args=['/usr/bin/python3', '-c', "import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())"], returncode=0)

## 1. Cấu hình dữ liệu
Sửa đường dẫn cho đúng cấu trúc bộ dữ liệu bạn Add vào (xem `/kaggle/input`).

In [8]:
from src.utils import load_yaml, save_yaml
from pathlib import Path

ds = load_yaml('configs/dataset.yaml')
# Set both paths explicitly if more than one dataset is attached.
IMAGES_DIR = "/kaggle/input/datasets/nhtnguye/dataminesapple/detection/detection/train/images"
MASKS_DIR = "/kaggle/input/datasets/nhtnguye/dataminesapple/detection/detection/train/masks"
if IMAGES_DIR is None or MASKS_DIR is None:
    candidates = [p for p in Path('/kaggle/input').rglob('images')
                  if p.is_dir() and (p.parent / 'masks').is_dir()
                  and 'test' not in p.parts and 'val' not in p.parts]
    train_candidates = [p for p in candidates if 'train' in p.parts]
    candidates = train_candidates or candidates
    if len(candidates) != 1:
        raise ValueError(f'Set IMAGES_DIR and MASKS_DIR explicitly. Found image/mask pairs: {candidates}')
    IMAGES_DIR = str(candidates[0])
    MASKS_DIR = str(candidates[0].parent / 'masks')
for folder in (IMAGES_DIR, MASKS_DIR):
    if not Path(folder).is_dir():
        raise FileNotFoundError(f'Dataset directory does not exist: {folder}')
ds['paths']['images_dir'] = IMAGES_DIR
ds['paths']['masks_dir'] = MASKS_DIR
ds['processed_dir'] = '/kaggle/working/processed'
ds.setdefault('yolo_export', {})['symlink'] = True  # Reuse images from /kaggle/input, no image copies.
save_yaml(ds, 'configs/dataset.yaml')
print(ds['paths'])


{'images_dir': '/kaggle/input/datasets/nhtnguye/dataminesapple/detection/detection/train/images', 'masks_dir': '/kaggle/input/datasets/nhtnguye/dataminesapple/detection/detection/train/masks', 'mask_suffix': '', 'labels_dir': None, 'coco_json': None}


In [9]:
import subprocess
import shlex
import sys

def run_python(arguments):
    subprocess.run([sys.executable, *shlex.split(arguments)], check=True)

run_python('scripts/prepare_data.py')

def run_model(model_key, arguments):
    config = load_yaml(f'configs/models/{model_key}.yaml')
    executable = config.get('python_executable') or sys.executable
    # Stream complete lines into the notebook output, including one line per epoch.
    command = [executable, '-u', *shlex.split(arguments)]
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1, errors='replace') as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        returncode = process.wait()
    if returncode:
        raise subprocess.CalledProcessError(returncode, command)
    from src.utils.resources import guard_disk
    usage = guard_disk()
    if usage:
        print(f"Working: {usage['used_gb']:.2f} GB used; {usage['remaining_budget_gb']:.2f} GB left in budget")



15:31:44 | train  470 ảnh | quả/ảnh: min=1  TB=42.2  max=121 | tổng=19837
15:31:44 | val    100 ảnh | quả/ảnh: min=1  TB=41.8  max=123 | tổng=4180
15:31:44 | test   100 ảnh | quả/ảnh: min=3  TB=41.6  max=117 | tổng=4159
15:31:45 | Đã xuất dataset YOLO: /kaggle/working/processed/yolo/data.yaml
15:31:45 | Đã lưu records/splits vào /kaggle/working/processed


## 2. Huấn luyện
Mặc định fine-tune một phần Grounding DINO, CountGD và VA-Count: train.scope = decoder_heads chỉ cập nhật decoder/head. RT-DETRv2 và RF-DETR cũng dùng chế độ này. Đổi scope thành full và dùng tên run mới nếu muốn train toàn bộ; không resume optimizer giữa hai chế độ. Không chọn tất cả model cùng lúc với ngân sách đĩa 18 GB.

CountGD tự tạo môi trường Python 3.10 / Transformers 4.x riêng và build CUDA ops; cần Kaggle GPU, Internet và CUDA toolkit. Grounding DINO dùng loss detector của Transformers; CountGD dùng criterion upstream; VA-Count là biến thể supervised density MSE + count loss với exemplar từ nhãn train, DINO chọn exemplar lúc test.

EPOCHS = None dùng số epoch của từng YAML; đặt 1 để chạy thử. Smoke test kiểm tra forward/backward và optimizer trước khi train. Checkpoint được lưu mỗi epoch để resume.

Push source/config lên GitHub và dùng checkout mới trên Kaggle; cell đầu không tự pull.

Resume phiên mới: điền RESUME_INPUT_DIR bên dưới. Input cần có runs/<name>/weights/last.pt (hoặc <name>/weights/last.pt); nên giữ cả best.pt, config.yaml và training/history.json. Notebook chép sang working rồi tự dùng --resume, không ghi vào Input và không ghi đè checkpoint working hiện có. Tên model và train.scope phải khớp phiên cũ. EPOCHS là tổng số epoch mục tiêu, không phải số epoch cộng thêm.


### Reports across sessions
`TRAIN_MODELS` controls training only. Set `RESUME_INPUT_DIRS` to previous Kaggle Outputs; saved models are discovered automatically. Keep the full runs directory including weights, training and evaluation.
Old test metrics are reused only for the same dataset and protocol. Set `REEVALUATE_OLD_MODELS=True` to evaluate again. Missing metrics trigger evaluation. Missing epoch history cannot be reconstructed from weights.
Set `GENERATE_REPORT_PREDICTIONS=False` to skip loading old models for sample images. Restored checkpoints count toward the disk budget. For overlapping sources, the existing local checkpoint wins.


In [ ]:
TRAIN_MODELS = ['grounding_dino', 'countgd', 'va_count']
RESUME_INPUT_DIRS = []  # Previous Kaggle Outputs containing runs/ or model folders.
REPORT_MODELS_EXTRA = []  # Optional extra config keys, e.g. pretrained models.
REEVALUATE_OLD_MODELS = False  # Reuse requires the same dataset and protocol.
GENERATE_REPORT_PREDICTIONS = True  # False skips loading old models for sample images.
# Chạy nhóm mới trong phiên riêng để phù hợp ngân sách đĩa.
# Các model khác: yolov8, csrnet, yolo11, faster_rcnn, mask_rcnn, rtdetrv2, rf_detr.

# Điền thư mục chứa runs hoặc chứa trực tiếp các thư mục model từ Kaggle Input.
RESUME_INPUT_DIR = ''  # Ví dụ: /kaggle/input/datasets/<user>/<dataset>

EPOCHS = None  # Dùng số epoch riêng trong YAML của từng model. Đặt 1 để chạy thử.

from pathlib import Path

LOW_MEMORY_MODE = True  # Batch 1, no DataLoader workers; keep model architecture unchanged.
SMOKE_TEST_NEW_MODELS = True  # Test real checkpoint before starting the long training loop.

# Checkpoints are saved after each epoch by the training code.
# To resume in a new Kaggle session, restore previous runs here first.
# Optional model keys: rtdetrv2, rf_detr, grounding_dino, countgd, va_count.
# The last two also require source/dependency setup described in Docs/Extra_Models.md.
from src.utils.report_sessions import discover_models, test_report
RUNS = Path('/kaggle/working/runs')
RUNS.mkdir(parents=True, exist_ok=True)
sources = list(dict.fromkeys([*RESUME_INPUT_DIRS, *([RESUME_INPUT_DIR] if globals().get('RESUME_INPUT_DIR', '').strip() else [])]))
for source in sources:
    if TRAIN_MODELS:
        subprocess.run([sys.executable, 'scripts/restore_runs.py', '--source', source,
                        '--out', str(RUNS), '--models', *TRAIN_MODELS], check=True)
    old_keys = [k for k in discover_models('configs/models', [source]) if k not in TRAIN_MODELS]
    if old_keys:
        subprocess.run([sys.executable, 'scripts/restore_runs.py', '--source', source,
                        '--out', str(RUNS), '--models', *old_keys, '--report-only'], check=True)
REPORT_MODELS = list(dict.fromkeys([*TRAIN_MODELS, *discover_models('configs/models', [RUNS]), *REPORT_MODELS_EXTRA]))
EVAL_MODELS = [k for k in REPORT_MODELS if k in TRAIN_MODELS or REEVALUATE_OLD_MODELS
               or test_report(RUNS / load_yaml(f'configs/models/{k}.yaml')['name']) is None]
ACTIVE_MODELS = list(dict.fromkeys([*TRAIN_MODELS, *EVAL_MODELS,
                                   *(REPORT_MODELS if GENERATE_REPORT_PREDICTIONS else [])]))
print('Train:', TRAIN_MODELS)
print('Report:', REPORT_MODELS)
print('Evaluate:', EVAL_MODELS)
print('Reuse test metrics:', [k for k in REPORT_MODELS if k not in EVAL_MODELS])

new_models = {'rtdetrv2', 'rf_detr', 'grounding_dino', 'va_count'}
if new_models.intersection(ACTIVE_MODELS):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-extra-models.txt'], check=True)
for research_model in ('countgd', 'va_count'):
    if research_model in ACTIVE_MODELS:
        run_python(f'scripts/setup_research_models.py --model {research_model}')

if 'countgd' in ACTIVE_MODELS:
    countgd_cfg = load_yaml('configs/models/countgd.yaml')
    countgd_python = countgd_cfg.get('python_executable')
    if not countgd_python or not Path(countgd_python).is_file():
        run_python('scripts/setup_countgd_env.py --env /kaggle/working/.venvs/countgd')

RUNS = Path('/kaggle/working/runs')
model_keys = ' '.join(TRAIN_MODELS)
if TRAIN_MODELS:
    run_python(f'scripts/check_resources.py --models {model_keys} --working-root /kaggle/working --budget-gb 18 --reserve-gb 2 --check-imports')
if SMOKE_TEST_NEW_MODELS:
    for model_key in TRAIN_MODELS:
        if model_key in {'rtdetrv2', 'rf_detr', 'grounding_dino', 'countgd', 'va_count'}:
            run_model(model_key, f'scripts/check_model.py --model configs/models/{model_key}.yaml --out /kaggle/working/runs --backward')
memory_args = '--batch-size 1 --num-workers 0' if LOW_MEMORY_MODE else ''
for m in TRAIN_MODELS:
    cfg = load_yaml(f'configs/models/{m}.yaml')
    if cfg.get('mode') == 'pretrained':
        print(f'{m}: chỉ suy luận pretrained; không huấn luyện')
        continue
    checkpoint = RUNS / cfg['name'] / 'weights' / 'last.pt'
    resume = '--resume' if checkpoint.is_file() else ''
    extra = f'--epochs {EPOCHS}' if EPOCHS else ''
    print(f'{m}: resume {checkpoint}' if resume else f'{m}: start training')
    run_model(m, f'scripts/train.py --model configs/models/{m}.yaml --out {RUNS} {extra} {resume} {memory_args}')


### Sai số đếm trên validation theo epoch
Dùng MAE (quả/ảnh) làm thước đo sai số chung cho các model. Đây là chỉ số validation, không phải loss tối ưu hóa khi huấn luyện: loss gốc của các kiến trúc khác nhau không so sánh trực tiếp được. Model pretrained không có lịch sử huấn luyện sẽ được bỏ qua.


In [ ]:
import json
import math
from pathlib import Path
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))
plotted = False
for m in REPORT_MODELS:
    cfg = load_yaml(f'configs/models/{m}.yaml')
    if cfg.get('mode') == 'pretrained':
        continue
    history_path = Path('/kaggle/working/runs') / cfg['name'] / 'training' / 'history.json'
    if not history_path.is_file():
        print(f"{cfg['name']}: chưa có history.json")
        continue
    with history_path.open(encoding='utf-8') as file:
        history = json.load(file)
    points = sorted(
        (int(row['epoch']) + 1, float(row['val_mae']))
        for row in history
        if row.get('val_mae') is not None and math.isfinite(float(row['val_mae']))
    )
    if not points:
        print(f"{cfg['name']}: chưa có val_mae")
        continue
    epochs, errors = zip(*points)
    ax.plot(epochs, errors, label=cfg['name'], linewidth=1.8)
    plotted = True

if plotted:
    ax.set(xlabel='Epoch', ylabel='Validation MAE (quả/ảnh) ↓',
           title='Sai số đếm trên validation theo epoch')
    ax.grid(alpha=0.25)
    ax.legend()
    fig.tight_layout()
    plt.show()
else:
    print('Chưa có lịch sử validation để vẽ đồ thị chung.')
plt.close(fig)


## 3. Đánh giá trên tập test

In [ ]:
for m in EVAL_MODELS:
    run_model(m, f'scripts/evaluate.py --model configs/models/{m}.yaml --out /kaggle/working/runs --split test --batch-size 1 --speed')

# YOLO + tiling (SAHI-style)
if 'yolov8' in EVAL_MODELS:
    run_python(f'scripts/evaluate.py --model configs/models/yolov8.yaml --out /kaggle/working/runs --split test --batch-size 1 --tile --tag _tile --speed')


14:21:31 | yolov8s: MAE=3.07 RMSE=4.73 MAPE=8.5% R2=0.972 mAP50=0.896 mAP50-95=0.485 FPS=33.1
14:21:31 | Đã ghi kết quả vào /kaggle/working/runs/yolov8s
14:22:18 | yolov8s: MAE=4.47 RMSE=6.36 MAPE=13.0% R2=0.950 mAP50=0.840 mAP50-95=0.441
14:22:18 | Đã ghi kết quả vào /kaggle/working/runs/yolov8s


In [ ]:
run_python('scripts/compare.py --runs /kaggle/working/runs --split test')

import pandas as pd
from IPython.display import display

comparison = pd.read_csv('/kaggle/working/runs/comparison/comparison_test.csv')
columns = ['model', 'mae', 'rmse', 'mape_pct', 'acc_within_3', 'fps', 'params_million', 'model_tensor_mb', 'gpu_peak_allocated_mb', 'checkpoint_mb']
summary = comparison.reindex(columns=columns).sort_values('mae').rename(columns={
    'model': 'Model',
    'mae': 'MAE ↓',
    'rmse': 'RMSE ↓',
    'mape_pct': 'MAPE ↓',
    'acc_within_3': 'Tỷ lệ sai số ≤ 3 quả ↑',
    'fps': 'Tốc độ (FPS) ↑',
    'params_million': 'Số tham số (M) ↓',
    'model_tensor_mb': 'Trọng số (MiB) ↓',
    'gpu_peak_allocated_mb': 'VRAM đỉnh (MiB) ↓',
    'checkpoint_mb': 'Checkpoint (MiB) ↓',
})
display(summary.style.format({
    'MAE ↓': '{:.2f}',
    'RMSE ↓': '{:.2f}',
    'MAPE ↓': '{:.2f}%',
    'Tỷ lệ sai số ≤ 3 quả ↑': '{:.1%}',
    'Tốc độ (FPS) ↑': '{:.1f}',
    'Số tham số (M) ↓': '{:.3f}',
    'Trọng số (MiB) ↓': '{:.1f}',
    'VRAM đỉnh (MiB) ↓': '{:.1f}',
    'Checkpoint (MiB) ↓': '{:.1f}',
}, na_rep='—').hide(axis='index').set_caption(
    'Kết quả trên tập test; MAE/RMSE tính bằng quả/ảnh. '
    'Dung lượng: MiB (1024² byte); trọng số gồm parameters và buffers; VRAM là bộ nhớ CUDA cấp phát đỉnh khi benchmark, — nếu không có dữ liệu. FPS chỉ so sánh trực tiếp khi đo cùng thiết bị và cấu hình.'
))
if {'benchmark_device', 'benchmark_hardware'}.issubset(comparison.columns):
    devices = comparison[['benchmark_device', 'benchmark_hardware']].drop_duplicates()
    if len(devices) > 1:
        print('Lưu ý: FPS được đo trên các thiết bị khác nhau: ' + '; '.join(
            f"{row.model}: {row.benchmark_device} / {row.benchmark_hardware}"
            for row in comparison.itertuples()
        ))


## 4. Ảnh dự đoán
Mỗi ảnh mẫu được trình bày riêng, tối đa 2 model trên một hàng để dễ quan sát. Đổi PREDICTION_COLUMNS thành 3 nếu muốn bố cục gọn hơn.


In [ ]:
import random
import shutil
from pathlib import Path
from src.data import load_split_records

records = load_split_records('/kaggle/working/processed', 'test')
samples = random.Random(0).sample(records, min(4, len(records)))
# A dedicated folder avoids mixing these samples with previous predictions.
sample_dir = Path('/kaggle/working/report_samples')
sample_dir.mkdir(parents=True, exist_ok=True)
for record in samples:
    shutil.copy(record.image_path, sample_dir / f'{record.image_id}.png')

for m in (REPORT_MODELS if GENERATE_REPORT_PREDICTIONS else []):
    run_model(m, f'scripts/predict.py --model configs/models/{m}.yaml --runs /kaggle/working/runs --source {sample_dir}')


In [ ]:
if GENERATE_REPORT_PREDICTIONS:
    import math
    import matplotlib.pyplot as plt
    import matplotlib.image as mpimg
    from IPython.display import Markdown, display

    PREDICTION_COLUMNS = 2  # Use 2 or 3 columns.
    if PREDICTION_COLUMNS not in (2, 3):
        raise ValueError('PREDICTION_COLUMNS must be 2 or 3')

    for record in samples:
        display(Markdown(f'**Ảnh: {record.image_id} — Số quả thực tế (GT): {record.count}**'))
        # Show one row at a time so notebook renderers keep every image large.
        for start in range(0, len(REPORT_MODELS), PREDICTION_COLUMNS):
            batch = REPORT_MODELS[start:start + PREDICTION_COLUMNS]
            images = []
            for m in batch:
                name = load_yaml(f'configs/models/{m}.yaml')['name']
                image_path = Path('/kaggle/working/runs') / name / 'predictions' / f'{record.image_id}_count.jpg'
                images.append((name, mpimg.imread(image_path) if image_path.is_file() else None))
            max_ratio = max((img.shape[0] / img.shape[1] for _, img in images if img is not None), default=1.5)
            fig, axes = plt.subplots(1, PREDICTION_COLUMNS,
                                     figsize=(12, 12 / PREDICTION_COLUMNS * max_ratio + 0.5),
                                     squeeze=False)
            for ax in axes[0]:
                ax.axis('off')
            for ax, (name, img) in zip(axes[0], images):
                if img is None:
                    ax.text(0.5, 0.5, 'Chưa có ảnh dự đoán', ha='center', va='center')
                else:
                    ax.imshow(img)
                ax.set_title(name, fontsize=14, pad=10)
            fig.subplots_adjust(left=0.01, right=0.99, bottom=0.01, top=0.95, wspace=0.04)
            plt.show()
            plt.close(fig)
